# 허깅페이스 허깅페이스에서 모델 받아 다국어 번역기 만들기
[facebook/m2m100_418M](https://huggingface.co/facebook/m2m100_418M)

In [2]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

text = "Trump postpones strike threat: US President Donald Trump told CNN there are 15 points of agreement between the US and Iran after talks this weekend. He announced he will hold off strikes against Iranian energy sites for five days, after earlier threatening an attack if Tehran did not let the Strait of Hormuz fully reopen. Oil prices dropped following Trump’s statement.• Iran responds: Iran’s foreign ministry said there was “no dialogue” between Tehran and Washington, according to state affiliated media. Separately, the semi-official Fars News Agency, citing what it described as informed Iranian sources, said plans are being prepared for potential actions targeting Tel Aviv and some regional allies of the US and Israel.• Growing toll: The number of people reported killed in Iran and Lebanon since the start of the conflict is now in the thousands."

model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

# translate Hindi to French
tokenizer.src_lang = "en"
encoded_hi = tokenizer(text, return_tensors="pt")
generated_tokens = model.generate(**encoded_hi, forced_bos_token_id=tokenizer.get_lang_id("ko"))
result1 = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
# => "La vie est comme une boîte de chocolat."

print(result1)

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


['트럼프는 공격 위협을 연기한다 : 미국 대통령 도널드 트럼프는 이번 주말 회담 후 미국과 이란 사이에 합의 15 포인트가 있다고 CNN에 말했다. 그는 이란의 에너지 시설에 대한 공격을 5 일 동안 중단 할 것이라고 발표 한 후 이전에 테헤란이 호르무즈 스트리트를 완전히 재개하지 않으면 공격을 위협했다. 석유 가격은 트럼프의 진술에 따라 떨어졌다.• 이란은 대답 : 이란 외무부는 테헤란과 워싱턴 사이에 "대화"가 없었다고 말했다.']


In [3]:
import gradio as gr
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

# 모델 로드 (처음 1번만 오래 걸림)
model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

# 번역 함수
def translate(text, src_lang, tgt_lang):
    if not text.strip():
        return ""
    
    tokenizer.src_lang = src_lang
    encoded = tokenizer(text, return_tensors="pt", truncation=True)

    generated_tokens = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.get_lang_id(tgt_lang)
    )

    result = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    return result[0]


# 언어 옵션
lang_dict = {
    "한국어": "ko",
    "영어": "en",
    "일본어": "ja",
    "중국어": "zh",
    "힌디어": "hi",
    "프랑스어": "fr"
}

# UI
with gr.Blocks() as demo:
    gr.Markdown("## 🌐 AI 번역기")

    with gr.Row():
        with gr.Column():
            src_lang = gr.Dropdown(
                choices=list(lang_dict.keys()),
                value="영어",
                label="입력 언어"
            )

            input_text = gr.Textbox(
                placeholder="번역할 내용을 입력하세요",
                lines=10
            )

        with gr.Column():
            tgt_lang = gr.Dropdown(
                choices=list(lang_dict.keys()),
                value="한국어",
                label="출력 언어"
            )

            output_text = gr.Textbox(
                placeholder="번역 결과",
                lines=10
            )

    translate_btn = gr.Button("번역하기")

    # 버튼 클릭 시 실행
    translate_btn.click(
        fn=lambda text, s, t: translate(text, lang_dict[s], lang_dict[t]),
        inputs=[input_text, src_lang, tgt_lang],
        outputs=output_text
    )

# 실행
demo.launch()

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [4]:
demo.close()

Closing server running on port: 7860


# 한영-영한 번역 결과를 음성으로 읽어주는 서비스

In [ ]:
# !pip install "transformers>=4.33" accelerate soundfile
# !pip install "kokoro>=0.9.2"
# !pip install uroman
# !sudo apt-get -y install espeak-ng

In [5]:
import gradio as gr
import torch
import soundfile as sf
import numpy as np

from transformers import (
    M2M100ForConditionalGeneration,
    M2M100Tokenizer,
    VitsModel,
    AutoTokenizer,
)
from kokoro import KPipeline   # Kokoro-82M 파이썬 래퍼
from uroman import Uroman      # 🔥 pip install uroman

# -----------------------------
# 0. 디바이스 (GPU 우선)
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# 1. 번역 모델 (M2M100)
# -----------------------------
translation_model_name = "facebook/m2m100_418M"
translation_model = M2M100ForConditionalGeneration.from_pretrained(
    translation_model_name
).to(device)
translation_tokenizer = M2M100Tokenizer.from_pretrained(translation_model_name)

# -----------------------------
# 2. 영어 TTS: Kokoro-82M
# -----------------------------
kokoro_pipeline = KPipeline(lang_code="a")

def tts_english_kokoro(text: str, path: str = "tts_en.wav") -> str | None:
    text = (text or "").strip()
    if not text:
        return None

    generator = kokoro_pipeline(text, voice="af_heart")

    chunks = []
    for _, _, audio in generator:
        chunks.append(audio)

    if not chunks:
        return None

    audio_cat = np.concatenate(chunks)
    sf.write(path, audio_cat, 24000)
    return path

# -----------------------------
# 3. 한국어 TTS: MMS-TTS-KOR (VITS)
#    🔥 pip uroman 사용
# -----------------------------
kor_tts_name = "facebook/mms-tts-kor"
kor_tts_model = VitsModel.from_pretrained(kor_tts_name).to(device)
kor_tts_tokenizer = AutoTokenizer.from_pretrained(kor_tts_name)

uroman = Uroman()   # 🔥 한 번만 초기화하면 됨

def tts_korean_mms(text: str, path: str = "tts_ko.wav") -> str | None:
    text = (text or "").strip()
    if not text:
        return None

    # 1) 🔥 uroman 로마자 변환 (찾는 발음형)
    roman_text = uroman.romanize_string(text)

    # 2) 토크나이저 입력
    inputs = kor_tts_tokenizer(roman_text, return_tensors="pt").to(device)

    # 3) 모델 인퍼런스
    with torch.no_grad():
        outputs = kor_tts_model(**inputs).waveform

    wav = outputs.squeeze().cpu().numpy()
    if wav.size == 0:
        return None

    sf.write(path, wav, samplerate=kor_tts_model.config.sampling_rate)
    return path

# -----------------------------
# 4. 언어 매핑
# -----------------------------
LANGUAGES = {
    "Korean": "ko",
    "English": "en",
    "Chinese": "zh",
    "Japanese": "ja",
    "French": "fr",
    "Spanish": "es",
    "German": "de",
    "Hindi": "hi",
}

# -----------------------------
# 5. 번역 + TTS 함수
# -----------------------------
def translate_and_speak(text, src_lang_name, tgt_lang_name):
    text = text or ""
    if not text.strip():
        return "", None

    src_lang = LANGUAGES[src_lang_name]
    tgt_lang = LANGUAGES[tgt_lang_name]

    # ----- 번역 -----
    if src_lang == tgt_lang:
        translated_text = text.strip()
    else:
        translation_tokenizer.src_lang = src_lang
        encoded = translation_tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(device)

        with torch.no_grad():
            generated_tokens = translation_model.generate(
                **encoded,
                forced_bos_token_id=translation_tokenizer.get_lang_id(tgt_lang),
                max_length=512,
            )

        translated_text = translation_tokenizer.batch_decode(
            generated_tokens, skip_special_tokens=True
        )[0]

    # ----- TTS -----
    audio_path = None
    if tgt_lang == "en":
        audio_path = tts_english_kokoro(translated_text)
    elif tgt_lang == "ko":
        audio_path = tts_korean_mms(translated_text)

    return translated_text, audio_path

# -----------------------------
# 6. Gradio UI
# -----------------------------
with gr.Blocks() as demo:
    gr.Markdown("## 🌐 다국어 번역 + TTS (영어 Kokoro + 한국어 MMS)")

    with gr.Row():
        src_lang = gr.Dropdown(list(LANGUAGES.keys()), label="원본 언어", value="Korean")
        tgt_lang = gr.Dropdown(list(LANGUAGES.keys()), label="번역 언어", value="English")

    with gr.Row():
        input_text = gr.Textbox(lines=8, label="입력 텍스트")
        output_text = gr.Textbox(lines=8, label="번역 결과", interactive=False)

    audio_output = gr.Audio(label="음성 출력", type="filepath")

    translate_button = gr.Button("번역 + 음성 생성")
    translate_button.click(
        fn=translate_and_speak,
        inputs=[input_text, src_lang, tgt_lang],
        outputs=[output_text, audio_output],
    )

    demo.queue()

demo.launch(inline=False, share=False)

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


config.json: 0.00B [00:00, ?B/s]

/home/mandongs/miniforge3/envs/hug/lib/python3.11/site-packages/torch/nn/modules/rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/home/mandongs/miniforge3/envs/hug/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


kokoro-v1_0.pth:   0%|          | 0.00/327M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/268 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
